## Q5 — Wins and Non-Wins Logic
The rubric asks for (1) cumulative wins and (2) cumulative completed races where the 
driver did NOT win. We filter to statusId = 1 (race completed) before computing 
counts, so DNFs are excluded from both tallies. A "non-win" is any completed race 
where finishing_position > 1. Both counts are cumulative up to and including the 
current race, using a rolling Window ordered chronologically per driver.

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
df_races   = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv",   header=True)
df_results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)

In [0]:
df_drivers_clean = df_drivers.select(
    "driverId",
    F.concat_ws(" ", F.col("forename"), F.col("surname")).alias("driver_name"),
    F.when(
        F.col("code").isNull() |
        (F.trim(F.col("code")) == "") |
        (F.col("code") == "\\N"),
        F.upper(F.substring(F.col("surname"), 1, 3))
    ).otherwise(F.col("code")).alias("driver_code")
)

In [0]:
df_results_clean = df_results.select(
    "raceId", "driverId",
    F.col("positionOrder").cast("int").alias("finishing_position"),
    F.col("statusId")
).filter(F.col("statusId") == "1")  # completed races only

In [0]:
df_joined = (
    df_results_clean
    .join(df_races.select("raceId", "year", F.col("name").alias("race_name")),
          on="raceId", how="left")
    .join(df_drivers_clean, on="driverId", how="left")
)

In [0]:
# ── 5. Binary flags ───────────────────────────────────────────────────────────
# is_win:     driver finished 1st
# is_non_win: driver completed the race but did NOT finish 1st
df_flagged = (
    df_joined
    .withColumn("is_win",     (F.col("finishing_position") == 1).cast("int"))
    .withColumn("is_non_win", (F.col("finishing_position") > 1).cast("int"))
)

In [0]:
#Cumulative counts up to and including each race
window_cumul = (
    Window.partitionBy("driverId")
          .orderBy("year", "raceId")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_q5 = (
    df_flagged
    .withColumn("cumul_wins",     F.sum("is_win").over(window_cumul))
    .withColumn("cumul_non_wins", F.sum("is_non_win").over(window_cumul))
    .select(
        "year", "race_name", "raceId",
        "driver_code", "driver_name",
        "finishing_position",
        "cumul_wins",
        "cumul_non_wins"
    )
    .orderBy("year", "raceId", "finishing_position")
)

display(df_q5)